# Fine-tuned transformers on LIAR (GPU)

Fine-tunes the content-based verdict models on LIAR (binary reliable / misleading), completing the comparison plan: **BERT (vanilla) vs RoBERTa (vanilla) vs RoBERTa (optimised)** — one architecture comparison, one fine-tuning comparison.

Same data pipeline and metrics as `src/factnet/nlp/` (CPU baselines for reference: TF-IDF + LogReg **0.633**, frozen DistilBERT **0.626** macro-F1).

**How to run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. About 20–30 min total. The last cell downloads `nlp_metrics.json`.

In [ ]:
%pip install -q transformers scikit-learn pandas tqdm

import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# LIAR download + binary target, identical to src/factnet/nlp/data.py
import io
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

LIAR_COLS = [
    "id", "label", "statement", "subject", "speaker", "job", "state", "party",
    "barely_true", "false", "half_true", "mostly_true", "pants_fire", "context",
]
MISLEADING = {"pants-fire", "false", "barely-true"}  # misleading (0) vs reliable (1)
ROOT = Path("/tmp/liar")
LIAR_URL = "https://www.cs.ucsb.edu/~william/data/liar_dataset.zip"

if not (ROOT / "train.tsv").exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(LIAR_URL) as response:
        payload = response.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        archive.extractall(ROOT)


def load_liar(split):
    df = pd.read_csv(ROOT / f"{split}.tsv", sep="\t", header=None, names=LIAR_COLS)
    df = df[["statement", "subject", "label"]].dropna(subset=["statement"]).copy()
    df = df.rename(columns={"statement": "text"})
    df["y"] = (~df["label"].isin(MISLEADING)).astype(int)  # 1 = reliable
    return df


splits = {s: load_liar(s) for s in ("train", "valid", "test")}
print({s: len(df) for s, df in splits.items()})

In [ ]:
# Fine-tuning loop: AdamW + linear schedule, best epoch selected on valid,
# final metrics reported on test (accuracy + macro-F1, as in the CPU baselines).
import transformers
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

transformers.logging.set_verbosity_error()


class LiarDataset(Dataset):
    def __init__(self, df, tok, max_len):
        enc = tok(df.text.tolist(), padding="max_length", truncation=True,
                  max_length=max_len, return_tensors="pt")
        self.ids, self.mask = enc["input_ids"], enc["attention_mask"]
        self.y = torch.tensor(df.y.values)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.ids[i], self.mask[i], self.y[i]


def linear_warmup(opt, warmup, total):
    def factor(step):
        if step < warmup:
            return step / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return torch.optim.lr_scheduler.LambdaLR(opt, factor)


def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for ids, mask, y in loader:
            logits = model(input_ids=ids.to(DEVICE),
                           attention_mask=mask.to(DEVICE)).logits
            ps += logits.argmax(dim=1).cpu().tolist()
            ys += y.tolist()
    return accuracy_score(ys, ps), f1_score(ys, ps, average="macro")


def finetune(model_name, tag, epochs=3, lr=2e-5, batch=32, max_len=64,
             warmup_frac=0.0, weight_decay=0.0, seed=0):
    print(f"\n=== {tag} ({model_name}) ===")
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2).to(DEVICE)
    loaders = {s: DataLoader(LiarDataset(splits[s], tok, max_len),
                             batch_size=batch, shuffle=(s == "train"))
               for s in splits}
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total = epochs * len(loaders["train"])
    sched = linear_warmup(opt, int(warmup_frac * total), total)

    best_f1, best_state = -1.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        for ids, mask, y in tqdm(loaders["train"], desc=f"epoch {epoch}", leave=False):
            opt.zero_grad()
            out = model(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE),
                        labels=y.to(DEVICE))
            out.loss.backward()
            opt.step()
            sched.step()
        acc, f1 = evaluate(model, loaders["valid"])
        print(f"  epoch {epoch}: valid acc {acc:.3f}  macro-F1 {f1:.3f}")
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    acc, f1 = evaluate(model, loaders["test"])
    print(f"  test: acc {acc:.3f}  macro-F1 {f1:.3f}")
    del model
    torch.cuda.empty_cache()
    return {"model": tag, "accuracy": round(acc, 3), "macro_f1": round(f1, 3)}

In [ ]:
results = [
    finetune("bert-base-uncased", "BERT (fine-tuned)"),
    finetune("roberta-base", "RoBERTa (vanilla fine-tune)"),
    finetune("roberta-base", "RoBERTa (optimised fine-tune)",
             epochs=4, warmup_frac=0.1, weight_decay=0.01, max_len=128),
]

In [ ]:
import json

baselines = [
    {"model": "TF-IDF + LogReg (CPU baseline)", "accuracy": 0.637, "macro_f1": 0.633},
    {"model": "DistilBERT frozen + LR (CPU baseline)", "accuracy": 0.631, "macro_f1": 0.626},
]

print(f"{'model':38s} {'accuracy':>9s} {'macro-F1':>9s}")
print("-" * 58)
for r in baselines + results:
    print(f"{r['model']:38s} {r['accuracy']:>9.3f} {r['macro_f1']:>9.3f}")

metrics = {"dataset": "LIAR (binary reliable / misleading)", "runs": results}
with open("nlp_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

try:
    from google.colab import files
    files.download("nlp_metrics.json")
except ImportError:
    print("saved nlp_metrics.json")